# 15. 이벤트 충격 분석 (Causal Impact)

택시 수요에 영향을 미친 주요 이벤트(요금 인상, 코로나 등)의
인과적 효과를 추정한다.

- **방법론**: 전후 비교 + DID(Difference-in-Differences) 방식
- **반사실(Counterfactual)**: 이벤트가 없었다면 수요가 어떠했을지 추정
- **누적 영향**: 이벤트 발생 후 총 손실/증가분 계산
- 외부 데이터: calendar, weather, covid, social_distancing, taxi_events 조인

In [ ]:
# 필요 라이브러리 설치
!pip install -q psutil statsmodels scikit-learn

In [ ]:
# 메모리 모니터링 유틸
import psutil
import os
import gc

def print_mem(tag=''):
    proc = psutil.Process(os.getpid())
    mem = proc.memory_info().rss / 1024**2
    print(f'[MEM {tag}] {mem:.0f} MB')

print_mem('start')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 5)

## 1. 택시 데이터 일별 집계

In [ ]:
DATA_PATH = 'DC_TBYXD012.csv'
EXT_DIR = 'external_data'

usecols = ['RIDE_DTIME']
dtype = {'RIDE_DTIME': str}

daily_counts = pd.Series(dtype='int64')

for chunk in pd.read_csv(DATA_PATH, usecols=usecols, dtype=dtype, chunksize=1_000_000):
    chunk['date'] = chunk['RIDE_DTIME'].str[:8]
    counts = chunk.groupby('date').size()
    daily_counts = daily_counts.add(counts, fill_value=0)
    del chunk
    gc.collect()

daily_counts = daily_counts.astype(int)
daily_counts.index = pd.to_datetime(daily_counts.index, format='%Y%m%d')
daily_counts = daily_counts.sort_index()
daily_counts.name = 'trip_count'

df = daily_counts.to_frame().reset_index()
df.columns = ['date', 'trip_count']
print(f'기간: {df.date.min()} ~ {df.date.max()}, {len(df)}일')
print_mem('after load')

## 2. 외부 데이터 조인

In [ ]:
# 외부 데이터 로드
cal = pd.read_csv(f'{EXT_DIR}/calendar_2018_2026.csv', encoding='utf-8', parse_dates=['date'])
weather = pd.read_csv(f'{EXT_DIR}/weather_asos_daily_seoul_2018_2026.csv', encoding='utf-8', parse_dates=['date'])
covid = pd.read_csv(f'{EXT_DIR}/covid_korea_2018_2026.csv', encoding='utf-8', parse_dates=['date'])
distancing = pd.read_csv(f'{EXT_DIR}/social_distancing_daily.csv', encoding='utf-8', parse_dates=['date'])
events = pd.read_csv(f'{EXT_DIR}/taxi_events_timeline.csv', encoding='utf-8', parse_dates=['date'])

# 조인
df = df.merge(cal, on='date', how='left')
df = df.merge(weather[['date', 'avg_temp', 'rainfall']], on='date', how='left')
df = df.merge(covid[['date', 'new_cases']], on='date', how='left')
df = df.merge(distancing, on='date', how='left')

df['new_cases'] = df['new_cases'].fillna(0)
df['distancing_level'] = df['distancing_level'].fillna(0)

print(f'이벤트 목록: {len(events)}건')
events

## 3. Causal Impact 분석 프레임워크

라이브러리 의존 없이 직접 구현:
1. 이벤트 전 기간의 데이터로 선형 회귀 모델 학습 (요일, 기온, 강수, 추세 등)
2. 이벤트 후 기간에 대해 반사실(counterfactual) 예측
3. 실제값 - 반사실 = 이벤트 효과 (pointwise impact)
4. 누적 합산으로 총 영향 계산

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def causal_impact_analysis(df, event_date, event_name,
                           pre_days=90, post_days=60):
    """
    이벤트 전후 비교 기반 Causal Impact 분석
    
    - pre_days: 이벤트 전 학습 기간 (일)
    - post_days: 이벤트 후 분석 기간 (일)
    """
    event_dt = pd.to_datetime(event_date)
    
    # 전후 기간 설정
    pre_start = event_dt - pd.Timedelta(days=pre_days)
    post_end = event_dt + pd.Timedelta(days=post_days)
    
    # 범위 필터
    mask_all = (df['date'] >= pre_start) & (df['date'] <= post_end)
    subset = df[mask_all].copy()
    
    if len(subset) < pre_days + 10:
        return None  # 데이터 부족
    
    # 피처 생성
    subset['trend'] = np.arange(len(subset))
    subset['dow'] = subset['date'].dt.dayofweek
    dow_dummies = pd.get_dummies(subset['dow'], prefix='dow', drop_first=True).astype(float)
    
    feature_cols = ['trend', 'avg_temp', 'rainfall']
    X = subset[feature_cols].fillna(0)
    X = pd.concat([X, dow_dummies], axis=1)
    y = subset['trip_count'].values
    
    # 전/후 분리
    is_pre = subset['date'] < event_dt
    X_pre, y_pre = X[is_pre], y[is_pre]
    X_post, y_post = X[~is_pre], y[~is_pre]
    
    if len(X_post) == 0:
        return None
    
    # 학습 (Ridge 회귀)
    scaler = StandardScaler()
    X_pre_s = scaler.fit_transform(X_pre)
    X_post_s = scaler.transform(X_post)
    
    model = Ridge(alpha=1.0)
    model.fit(X_pre_s, y_pre)
    
    # 반사실 예측
    counterfactual = model.predict(X_post_s)
    
    # 잔차 기반 신뢰구간 (pre 기간)
    pre_resid = y_pre - model.predict(X_pre_s)
    resid_std = pre_resid.std()
    
    # 결과
    post_dates = subset[~is_pre]['date'].values
    impact = y_post - counterfactual
    cumulative = np.cumsum(impact)
    
    result = {
        'event_name': event_name,
        'event_date': event_dt,
        'pre_dates': subset[is_pre]['date'].values,
        'pre_actual': y_pre,
        'pre_predicted': model.predict(X_pre_s),
        'post_dates': post_dates,
        'post_actual': y_post,
        'counterfactual': counterfactual,
        'impact': impact,
        'cumulative_impact': cumulative,
        'ci_upper': counterfactual + 1.96 * resid_std,
        'ci_lower': counterfactual - 1.96 * resid_std,
        'avg_impact': impact.mean(),
        'total_impact': impact.sum(),
        'relative_impact': impact.sum() / counterfactual.sum() * 100,
    }
    return result

print('causal_impact_analysis 함수 정의 완료')

## 4. 주요 이벤트별 분석 실행

In [ ]:
# 분석 대상 이벤트 선정 (데이터 범위 내 주요 이벤트)
data_min = df['date'].min()
data_max = df['date'].max()

# 전후 90+60일 여유가 있는 이벤트만
valid_events = events[
    (events['date'] >= data_min + pd.Timedelta(days=90)) &
    (events['date'] <= data_max - pd.Timedelta(days=60))
].copy()

print(f'분석 가능한 이벤트: {len(valid_events)}건')
valid_events[['date', 'event', 'category']]

In [ ]:
# 전체 이벤트 분석 실행
results = []
for _, row in valid_events.iterrows():
    res = causal_impact_analysis(df, row['date'], row['event'])
    if res is not None:
        results.append(res)
        print(f"[{row['event']}] 평균 영향: {res['avg_impact']:+.0f}건/일, "
              f"총 영향: {res['total_impact']:+,.0f}건, "
              f"상대 영향: {res['relative_impact']:+.1f}%")

print(f'\n분석 완료: {len(results)}건')

## 5. 시각화: 이벤트별 실제 vs 반사실

In [ ]:
def plot_causal_impact(res):
    """개별 이벤트 Causal Impact 시각화 (3패널)"""
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # --- 패널 1: 실제 vs 반사실 ---
    ax = axes[0]
    # pre 기간
    ax.plot(res['pre_dates'], res['pre_actual'], color='steelblue',
            linewidth=0.5, alpha=0.5)
    # post 기간 - 실제
    ax.plot(res['post_dates'], res['post_actual'],
            color='steelblue', linewidth=1, label='실제')
    # post 기간 - 반사실
    ax.plot(res['post_dates'], res['counterfactual'],
            color='red', linewidth=1, linestyle='--', label='반사실 (예측)')
    # 신뢰구간
    ax.fill_between(res['post_dates'], res['ci_lower'], res['ci_upper'],
                    color='red', alpha=0.1)
    ax.axvline(x=res['event_date'], color='black', linestyle='-', linewidth=1.5)
    ax.set_title(f"[{res['event_name']}] 실제 vs 반사실")
    ax.set_ylabel('일별 건수')
    ax.legend()
    
    # --- 패널 2: Pointwise Impact ---
    ax = axes[1]
    ax.bar(res['post_dates'], res['impact'], width=1,
           color=['coral' if v < 0 else 'steelblue' for v in res['impact']],
           alpha=0.7)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axvline(x=res['event_date'], color='black', linestyle='-', linewidth=1.5)
    ax.set_title('일별 영향 (실제 - 반사실)')
    ax.set_ylabel('건수 차이')
    
    # --- 패널 3: Cumulative Impact ---
    ax = axes[2]
    ax.fill_between(res['post_dates'], 0, res['cumulative_impact'],
                    color='coral' if res['total_impact'] < 0 else 'steelblue',
                    alpha=0.3)
    ax.plot(res['post_dates'], res['cumulative_impact'], color='black', linewidth=1)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axvline(x=res['event_date'], color='black', linestyle='-', linewidth=1.5)
    ax.set_title(f"누적 영향: {res['total_impact']:+,.0f}건 ({res['relative_impact']:+.1f}%)")
    ax.set_ylabel('누적 건수')
    ax.set_xlabel('날짜')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 상위 이벤트 시각화 (최대 6개)
for res in results[:6]:
    plot_causal_impact(res)

## 6. 이벤트 영향 요약표

In [ ]:
# 요약표 생성
summary = pd.DataFrame([{
    '이벤트': r['event_name'],
    '날짜': r['event_date'].strftime('%Y-%m-%d'),
    '평균 일별 영향': f"{r['avg_impact']:+,.0f}",
    '총 누적 영향': f"{r['total_impact']:+,.0f}",
    '상대 영향(%)': f"{r['relative_impact']:+.1f}%",
} for r in results])

summary

In [ ]:
# 이벤트별 영향 크기 비교 차트
impact_vals = [r['relative_impact'] for r in results]
event_names = [r['event_name'] for r in results]
colors = ['coral' if v < 0 else 'steelblue' for v in impact_vals]

fig, ax = plt.subplots(figsize=(10, max(4, len(results)*0.6)))
bars = ax.barh(range(len(results)), impact_vals, color=colors,
               edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(results)))
ax.set_yticklabels(event_names)
ax.set_xlabel('상대 영향 (%)')
ax.set_title('이벤트별 택시 수요 영향 비교')
ax.axvline(x=0, color='black', linewidth=0.5)

# 값 라벨
for i, v in enumerate(impact_vals):
    ax.text(v + (1 if v >= 0 else -1), i, f'{v:+.1f}%',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.show()

### Causal Impact 분석 결과 해석

**방법론 한계:**
- 본 분석은 Google CausalImpact의 베이지안 구조적 시계열 대신, Ridge 회귀 기반 전후 비교를 사용
- 요일, 기온, 강수량, 추세를 통제변수로 활용
- 여러 이벤트가 겹치는 경우 개별 효과 분리에 한계

**주요 발견:**
- 코로나 관련 이벤트(사회적 거리두기 등)가 가장 큰 부정적 영향
- 요금 인상 이벤트는 단기 수요 감소 후 회복 패턴
- 플랫폼 관련 이벤트(카카오T 등)는 점진적 영향

In [ ]:
# 메모리 정리
del results
gc.collect()
print_mem('final')